# T1 · Construção automática de panoramas
**MO446 — Visão Computacional**

O objetivo é construir um panorama a partir de uma pasta sem ordem conhecida, identificar uma imagem de outra cena e reduzir fantasmas nas sobreposições. As seis etapas seguem o [enunciado](T1.pdf). O código usa OpenCV para características e estimação robusta; grafo, DLT didático, seleção de fontes e pirâmides ficam explícitos.

A coleta contém sete vistas noturnas de uma rua. Uma delas aponta mais para cima e continua sobreposta às demais. Uma fotografia externa da NASA foi acrescentada como controle de rejeição. Os resultados abaixo são calculados durante a execução, sem lista de vizinhos ou rótulo de intrusa na montagem.

**Leitura:** cada etapa reúne uma ideia, as funções correspondentes e um experimento. Para reproduzir, execute todas as células ou rode `python executar.py --entrada images/v1`. As referências completas estão ao final e em [REFERENCIAS.md](REFERENCIAS.md).

In [ ]:
PARAMETROS = {'entrada': 'images/v1', 'saida': 'outputs', 'resolucao': 1600}

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from io import BytesIO
import hashlib
import itertools
import json
import time
import csv
import tempfile

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tifffile
from scipy.sparse.csgraph import minimum_spanning_tree, shortest_path
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation

cv.setNumThreads(1)
plt.rcParams.update({'font.size': 10, 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight'})

@dataclass
class Config:
    resolucao: int = 1600
    pontos: int = 5000
    ratio: float = 0.80
    ransac_px: float = 3.0
    min_inliers: int = 25
    min_taxa: float = 0.20
    min_cobertura: float = 0.004
    niveis: int = 5
    seed: int = 446

@dataclass
class Quadro:
    uid: str
    nome: str
    rgb: np.ndarray
    fonte: str
    tamanho_original: tuple
    escala: float

@dataclass
class Caracteristicas:
    pontos: list
    descritores: np.ndarray
    segundos: float

@dataclass
class Par:
    i: int
    j: int
    H: np.ndarray | None
    origem: np.ndarray
    destino: np.ndarray
    inliers: np.ndarray
    indices: np.ndarray
    razoes: np.ndarray
    candidatos: np.ndarray
    erros: np.ndarray
    cobertura: float
    aceito: bool

    @property
    def n(self):
        return int(self.inliers.sum())

    @property
    def taxa(self):
        return self.n / len(self.inliers) if len(self.inliers) else 0.0


from dataclasses import asdict, replace
from IPython.display import display, HTML, Markdown, Image as ImagemNotebook
import html
import platform
import importlib.metadata

In [ ]:
def tabela(linhas, colunas=None):
    if not linhas:
        return
    colunas = colunas or list(linhas[0])
    def valor(v):
        return f'{v:.3f}' if isinstance(v, (float, np.floating)) else str(v)
    cabecalho = ''.join(f'<th>{html.escape(c)}</th>' for c in colunas)
    corpo = ''.join('<tr>' + ''.join(f'<td>{html.escape(valor(l[c]))}</td>' for c in colunas) + '</tr>' for l in linhas)
    display(HTML('<table><thead><tr>'+cabecalho+'</tr></thead><tbody>'+corpo+'</tbody></table>'))


def figura(fig, nome):
    caminho = FIGURAS / f'{nome}.jpg'
    fig.savefig(caminho, dpi=130, facecolor='white', pil_kwargs={'quality': 90})
    plt.close(fig)
    display(ImagemNotebook(filename=str(caminho)))


def painel(imagens, titulos, nome, colunas=4, altura=7):
    linhas = int(np.ceil(len(imagens)/colunas))
    fig, axes = plt.subplots(linhas, colunas, figsize=(13, altura), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, img, titulo in zip(axes.flat, imagens, titulos):
        ax.imshow(img)
        ax.set_title(titulo, fontsize=10)
    fig.tight_layout()
    figura(fig, nome)


def desenho_matches(quadros, features, par, ids, limite=70):
    amostra = ids[np.linspace(0, len(ids)-1, min(limite, len(ids))).astype(int)] if len(ids) else []
    matches = [cv.DMatch(int(a), int(b), 0.) for a,b in amostra]
    return cv.drawMatches(quadros[par.i].rgb, features[par.i].pontos,
                          quadros[par.j].rgb, features[par.j].pontos, matches, None,
                          matchColor=(25,210,160), singlePointColor=(255,180,40),
                          flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)


def serializavel(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(type(obj).__name__)

In [ ]:
cfg = Config(resolucao=int(PARAMETROS['resolucao']))
SAIDA = Path(PARAMETROS['saida'])
FIGURAS = SAIDA / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
print(f'Python {platform.python_version()} · OpenCV {cv.__version__} · NumPy {np.__version__}')
tabela([asdict(cfg)])

## 1. Coleta e leitura

As sete imagens foram feitas com um **Galaxy S24 Ultra**, de uma sacada voltada para uma rua, à noite. Há fachadas, fios, vegetação e um carro que muda entre as vistas. A iluminação urbana permite extrair detalhes, mas céu escuro, reflexos e saturação das luminárias tornam a coleta mais difícil que uma cena diurna. A cidade e os integrantes do grupo não foram informados.

Os DNG têm 8160 × 6120 pixels antes da orientação e incluem um JPEG renderizado pela câmera. A leitura abaixo extrai esse JPEG, aplica a orientação e limita o maior lado a 1600 pixels. **Este experimento não revela os dados RAW de 16 bits.** A tentativa com LibRaw falhou para a compressão destes arquivos; extrair a imagem incorporada evita uma conversão de cor não validada. [Adobe DNG](https://helpx.adobe.com/ca/camera-raw/desktop/dng-and-file-formats/digital-negative.html), [tifffile](https://github.com/cgohlke/tifffile).

O identificador deriva dos pixels reduzidos. A ordenação inicial pelo hash serve apenas para repetir desempates e sementes: não contém informação espacial. Nomes e horários não entram na geometria. A foto de Eileen Collins, da NASA, está em domínio público segundo a [documentação do scikit-image](https://scikit-image.org/docs/0.20.x/api/skimage.data.html#skimage.data.astronaut); ela é um controle externo, não uma foto da coleta.

In [ ]:
def ler_rgb(caminho):
    caminho = Path(caminho)
    if caminho.suffix.lower() == '.dng':
        with tifffile.TiffFile(caminho) as arquivo:
            principal = arquivo.pages[0]
            paginas = list(arquivo.pages)
            if principal.pages is not None:
                paginas += list(principal.pages)
            candidatos = [p for p in paginas if int(p.compression) == 7 and len(p.dataoffsets) == 1]
            if not candidatos:
                raise ValueError(f'DNG sem JPEG incorporado compatível: {caminho.name}')
            pagina = max(candidatos, key=lambda p: p.imagewidth * p.imagelength)
            with caminho.open('rb') as stream:
                stream.seek(pagina.dataoffsets[0])
                dados = stream.read(pagina.databytecounts[0])
            img = Image.open(BytesIO(dados)).convert('RGB')
            orientacao = int(principal.tags['Orientation'].value) if 'Orientation' in principal.tags else 1
            rotacoes = {2: Image.Transpose.FLIP_LEFT_RIGHT, 3: Image.Transpose.ROTATE_180,
                        4: Image.Transpose.FLIP_TOP_BOTTOM, 5: Image.Transpose.TRANSPOSE,
                        6: Image.Transpose.ROTATE_270, 7: Image.Transpose.TRANSVERSE,
                        8: Image.Transpose.ROTATE_90}
            if orientacao in rotacoes:
                img = img.transpose(rotacoes[orientacao])
        return img, 'jpeg_incorporado'
    with Image.open(caminho) as original:
        img = ImageOps.exif_transpose(original).convert('RGB')
    return img, 'rgb'


def carregar_imagens(pasta, cfg):
    caminhos = [p for p in Path(pasta).iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.dng', '.tif', '.tiff'}]
    if not caminhos:
        raise ValueError('A pasta não contém imagens reconhecidas.')
    quadros = []
    for caminho in caminhos:
        img, fonte = ler_rgb(caminho)
        original = img.size
        escala = min(1.0, cfg.resolucao / max(original))
        if escala < 1:
            img = img.resize(tuple(round(v * escala) for v in original), Image.Resampling.LANCZOS)
        rgb = np.asarray(img).copy()
        uid = hashlib.sha256(rgb.tobytes()).hexdigest()[:12]
        quadros.append(Quadro(uid, caminho.name, rgb, fonte, original, escala))
    if len({q.uid for q in quadros}) != len(quadros):
        raise ValueError('Há imagens idênticas na resolução de trabalho.')
    return sorted(quadros, key=lambda q: q.uid)

In [ ]:
quadros = carregar_imagens(PARAMETROS['entrada'], cfg)
rotulos = [f'I{i+1:02}' for i in range(len(quadros))]
inventario = [{'id':rotulos[i], 'arquivo':q.nome, 'uid_pixels':q.uid, 'fonte':q.fonte,
               'original':str(q.tamanho_original), 'trabalho':str(q.rgb.shape[1::-1])}
              for i,q in enumerate(quadros)]
tabela(inventario)
salvar_inventario = SAIDA / 'inventario.json'
salvar_inventario.write_text(json.dumps(inventario, ensure_ascii=False, indent=2))
painel([q.rgb for q in quadros], rotulos, '01_entrada', altura=9)

## 2. Detecção e descrição de características

Um detector localiza pontos; um descritor representa a vizinhança para compará-la em outra imagem. **SIFT** procura extremos em espaço de escala e descreve gradientes orientados [R1]. **ORB** combina FAST orientado e um descritor binário derivado do BRIEF [R2].

Usamos a mesma entrada e orçamento nominal de 5000 pontos por imagem. Os tempos excluem leitura, conversão para cinza e gráficos: são medianas de três extrações após aquecimento, com uma thread do OpenCV. A quantidade de pontos não mede, sozinha, a qualidade do alinhamento.

Nas figuras, círculos indicam escala e traços indicam orientação. Apenas 250 pontos de maior resposta aparecem por imagem, para manter a leitura; os descritores completos são usados no matching.

In [ ]:
def extrair(quadros, metodo, cfg):
    detector = cv.SIFT_create(nfeatures=cfg.pontos) if metodo == 'SIFT' else cv.ORB_create(nfeatures=cfg.pontos, WTA_K=2)
    saida = []
    detector.detectAndCompute(cv.cvtColor(quadros[0].rgb, cv.COLOR_RGB2GRAY), None)
    for quadro in quadros:
        cinza = cv.cvtColor(quadro.rgb, cv.COLOR_RGB2GRAY)
        tempos = []
        for _ in range(3):
            inicio = time.perf_counter()
            pontos, descritores = detector.detectAndCompute(cinza, None)
            tempos.append(time.perf_counter() - inicio)
        saida.append(Caracteristicas(pontos, descritores, float(np.median(tempos))))
    return saida

In [ ]:
features = {metodo: extrair(quadros, metodo, cfg) for metodo in ['SIFT','ORB']}
medidas_features = [{'metodo':m, 'imagem':rotulos[i], 'pontos':len(f.pontos),
                     'extracao_ms':1000*f.segundos} for m,fs in features.items() for i,f in enumerate(fs)]
tabela(medidas_features)
for metodo, fs in features.items():
    desenhos = []
    for q, f in zip(quadros, fs):
        pontos = sorted(f.pontos, key=lambda p: -p.response)[:250]
        desenhos.append(cv.drawKeypoints(q.rgb, pontos, None, color=(40,230,150),
                                        flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS))
    painel(desenhos, [f'{r} · {len(f.pontos)} pontos' for r,f in zip(rotulos,fs)],
           f'02_keypoints_{metodo.lower()}', altura=9)

## 3. Emparelhamento e ratio test

Para cada descritor, a busca exata retorna os dois vizinhos mais próximos. A distância é L2 para SIFT e Hamming para ORB (`WTA_K=2`). Mantemos o primeiro candidato quando

\[
\frac{d_1}{d_2}<\tau,\qquad \tau=0{,}80.
\]

O teste descarta comparações ambíguas, como padrões repetidos em janelas e grades. Lowe usou 0,8 em seu experimento [R1, §7.1]; não é um limiar universal. A próxima etapa mede a sensibilidade nesta coleta. Casos sem descritores ou sem dois vizinhos não produzem correspondência. [OpenCV: matching](https://docs.opencv.org/4.13.0/dc/dc3/tutorial_py_matcher.html).

A verificação geométrica já é calculada aqui, pois o grafo da etapa 4 exige **inliers**. A homografia e o RANSAC são desenvolvidos na etapa 5. Além da quantidade, exigimos proporção de inliers, distribuição espacial e transformação plausível. Esses limites são escolhas experimentais registradas na configuração.

In [ ]:
def projetar(H, pontos):
    pontos = np.asarray(pontos, float)
    p = np.c_[pontos, np.ones(len(pontos))] @ np.asarray(H, float).T
    if np.any(np.abs(p[:, 2]) < 1e-10):
        raise ValueError('Projeção com ponto no infinito.')
    return p[:, :2] / p[:, 2, None]


def cantos(imagem):
    h, w = imagem.shape[:2]
    return np.array([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]], float)


def homografia_plausivel(H, imagem):
    if H is None or not np.isfinite(H).all() or abs(np.linalg.det(H)) < 1e-12:
        return False
    c = cantos(imagem)
    denominadores = np.c_[c, np.ones(4)] @ H[2]
    if np.min(denominadores) * np.max(denominadores) <= 0:
        return False
    try:
        area = abs(cv.contourArea(projetar(H, c).astype(np.float32)))
    except ValueError:
        return False
    return 0.05 < area / (imagem.shape[0]*imagem.shape[1]) < 20


def emparelhar(quadros, features, metodo, cfg):
    matcher = cv.BFMatcher(cv.NORM_L2 if metodo == 'SIFT' else cv.NORM_HAMMING)
    pares = []
    for i, j in itertools.combinations(range(len(quadros)), 2):
        a, b = features[i], features[j]
        indices, razoes, candidatos = [], [], []
        if a.descritores is not None and b.descritores is not None and len(b.descritores) >= 2:
            for vizinhos in matcher.knnMatch(a.descritores, b.descritores, k=2):
                if len(vizinhos) < 2:
                    continue
                primeiro, segundo = vizinhos
                ratio = primeiro.distance / segundo.distance if segundo.distance > 0 else 1.0
                candidatos.append((primeiro.queryIdx, primeiro.trainIdx))
                razoes.append(ratio)
                if primeiro.distance < cfg.ratio * segundo.distance:
                    indices.append((primeiro.queryIdx, primeiro.trainIdx))
        indices = np.array(indices, int).reshape(-1, 2)
        origem = np.array([a.pontos[k].pt for k in indices[:, 0]], np.float32).reshape(-1, 2)
        destino = np.array([b.pontos[k].pt for k in indices[:, 1]], np.float32).reshape(-1, 2)
        H, mascara = None, np.zeros(len(indices), bool)
        if len(indices) >= 4:
            seed = int(hashlib.sha256((quadros[i].uid+quadros[j].uid).encode()).hexdigest()[:7], 16)
            cv.setRNGSeed(seed)
            H, m = cv.findHomography(origem, destino, cv.RANSAC, cfg.ransac_px, maxIters=10000, confidence=.999)
            if m is not None:
                mascara = m.ravel().astype(bool)
        erros = np.full(len(indices), np.inf)
        cobertura = 0.
        if H is not None:
            try:
                erros = np.linalg.norm(projetar(H, origem) - destino, axis=1)
            except ValueError:
                H = None
            if mascara.sum() >= 4:
                areas = [cv.contourArea(cv.convexHull(p[mascara])) / (q.rgb.shape[0]*q.rgb.shape[1])
                         for p, q in [(origem, quadros[i]), (destino, quadros[j])]]
                cobertura = min(areas)
        aceito = (int(mascara.sum()) >= cfg.min_inliers and mascara.mean() >= cfg.min_taxa
                  and cobertura >= cfg.min_cobertura and homografia_plausivel(H, quadros[i].rgb)) if len(mascara) else False
        pares.append(Par(i, j, H, origem, destino, mascara, indices, np.array(razoes),
                         np.array(candidatos, int).reshape(-1, 2), erros, cobertura, bool(aceito)))
    return pares

In [ ]:
pares_por_metodo = {m: emparelhar(quadros, fs, m, cfg) for m,fs in features.items()}
pares = pares_por_metodo['SIFT']
par_exemplo = max((p for p in pares if p.aceito), key=lambda p:p.n)
fig, axes = plt.subplots(3,1,figsize=(10,15))
for ax, ids, titulo in zip(axes,
    [par_exemplo.candidatos, par_exemplo.indices, par_exemplo.indices[par_exemplo.inliers]],
    ['Vizinho mais próximo', 'Após ratio test', 'Inliers de RANSAC']):
    ax.imshow(desenho_matches(quadros, features['SIFT'], par_exemplo, ids))
    ax.set_title(f'{titulo} · {len(ids)} correspondências (até 70 desenhadas)')
    ax.axis('off')
fig.tight_layout()
figura(fig, '03_matches')
fig, ax = plt.subplots(figsize=(8,3))
ax.hist(par_exemplo.razoes, bins=40, color='#2b788c')
ax.axvline(cfg.ratio, color='#bf573f', label=f'τ = {cfg.ratio:.2f}')
ax.set(xlabel='Razão entre as duas menores distâncias', ylabel='Descritores')
ax.legend()
figura(fig, '03_ratio')

## 4. Conectividade, rejeição e ordem espacial

Cada imagem é um vértice. Uma aresta aceita recebe o número de inliers do par. Calculamos uma direção por par, definida pelos identificadores de pixels, e espelhamos esse suporte na matriz; não somamos duas estimativas direcionais.

Selecionamos o maior componente conectado, como reconhecimento de um panorama [R5, §3]. A referência minimiza a soma das distâncias no grafo, usando custo inverso ao suporte. Uma árvore geradora de máximo peso fornece as homografias iniciais. Os centros transformados são ordenados no eixo principal da varredura; depois do ajuste de câmeras, usamos seus ângulos horizontais. São escolhas deste projeto para uma varredura horizontal, e não recuperação da cronologia.

A rejeição significa ausência de conexão suficiente com o componente principal. Uma vista válida com pouca textura também pode ser rejeitada. A foto apontada para cima deve permanecer se suas correspondências forem consistentes; o algoritmo não conhece um rótulo manual de intrusa.

In [ ]:
def conectividade(n, pares):
    matriz = np.zeros((n, n), float)
    for par in pares:
        if par.aceito:
            matriz[par.i, par.j] = matriz[par.j, par.i] = par.n
    return matriz


def componentes(matriz):
    restantes = set(range(len(matriz)))
    grupos = []
    while restantes:
        grupo, fila = set(), [min(restantes)]
        while fila:
            i = fila.pop()
            if i in grupo:
                continue
            grupo.add(i)
            fila.extend(j for j in np.flatnonzero(matriz[i]) if j not in grupo)
        restantes -= grupo
        grupos.append(sorted(grupo))
    return sorted(grupos, key=lambda g: (-len(g), -matriz[np.ix_(g, g)].sum(), g))


def alinhar_grafo(quadros, pares):
    matriz = conectividade(len(quadros), pares)
    grupo = componentes(matriz)[0]
    if len(grupo) < 2:
        raise ValueError('Nenhum par tem suporte geométrico suficiente.')
    sub = matriz[np.ix_(grupo, grupo)]
    dist = shortest_path(np.where(sub > 0, 1 / np.maximum(sub, 1), 0), directed=False)
    referencia = grupo[int(np.argmin(dist.sum(axis=1)))]
    arvore = minimum_spanning_tree(-sub).toarray()
    adj = {i: [] for i in grupo}
    for a, b in zip(*np.nonzero(arvore)):
        i, j = grupo[a], grupo[b]
        adj[i].append(j)
        adj[j].append(i)
    por_par = {(p.i, p.j): p for p in pares}
    transforms, usados = {referencia: np.eye(3)}, []
    fila = [referencia]
    while fila:
        pai = fila.pop(0)
        for filho in sorted(adj[pai]):
            if filho in transforms:
                continue
            p = por_par[tuple(sorted((pai, filho)))]
            H = p.H if p.i == filho else np.linalg.inv(p.H)
            transforms[filho] = transforms[pai] @ H
            transforms[filho] /= transforms[filho][2, 2]
            usados.append((pai, filho))
            fila.append(filho)
    centros = np.array([projetar(transforms[i], [[quadros[i].rgb.shape[1]/2, quadros[i].rgb.shape[0]/2]])[0] for i in grupo])
    _, _, vt = np.linalg.svd(centros - centros.mean(axis=0), full_matrices=False)
    eixo = vt[0] * (1 if vt[0, 0] >= 0 else -1)
    ordem = [grupo[k] for k in np.argsort(centros @ eixo)]
    if any(matriz[i, j] == 0 for i, j in zip(ordem, ordem[1:])):
        raise ValueError('A ordem espacial não forma uma sequência conectada.')
    return matriz, grupo, referencia, transforms, usados, ordem

In [ ]:
W, grupo, referencia, H, arestas, ordem = alinhar_grafo(quadros, pares)
comparacao = []
for metodo, fs in features.items():
    pp = pares_por_metodo[metodo]
    aceitos = [p for p in pp if p.aceito]
    erros = np.concatenate([p.erros[p.inliers] for p in aceitos])
    comparacao.append({'metodo':metodo, 'pontos':sum(len(f.pontos) for f in fs),
        'extracao_total_ms':1000*sum(f.segundos for f in fs), 'arestas':len(aceitos),
        'maior_componente':len(componentes(conectividade(len(quadros),pp))[0]),
        'inliers_arestas':sum(p.n for p in aceitos),
        'taxa_ponderada':sum(p.n for p in aceitos)/sum(len(p.inliers) for p in aceitos),
        'erro_medio_px':float(erros.mean())})
tabela(comparacao)
sensibilidade = []
for tau in [.70,.75,.80,.85]:
    pp = pares if tau == cfg.ratio else emparelhar(quadros, features['SIFT'], 'SIFT', replace(cfg,ratio=tau))
    grupos_tau = componentes(conectividade(len(quadros),pp))
    sensibilidade.append({'ratio':tau, 'arestas':sum(p.aceito for p in pp),
                          'tamanhos_componentes':str([len(g) for g in grupos_tau])})
tabela(sensibilidade)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
im = axes[0].imshow(W, cmap='Blues')
axes[0].set(xticks=range(len(quadros)), yticks=range(len(quadros)),
            xticklabels=rotulos, yticklabels=rotulos, title='Suporte das arestas aceitas')
for i in range(len(quadros)):
    for j in range(len(quadros)):
        axes[0].text(j,i,str(int(W[i,j])),ha='center',va='center',fontsize=8,
                     color='white' if W[i,j]>W.max()*.55 else '#182c3a')
fig.colorbar(im, ax=axes[0], label='Inliers', shrink=.8)
a = np.linspace(0,2*np.pi,len(quadros),endpoint=False)
pos = np.c_[np.cos(a),np.sin(a)]
for p in pares:
    if p.aceito:
        x,y=pos[[p.i,p.j]].T
        axes[1].plot(x,y,color='#9eb6be',lw=.5+2*p.n/W.max())
        axes[1].text(x.mean(),y.mean(),str(p.n),fontsize=8,backgroundcolor='white',ha='center')
for i,(x,y) in enumerate(pos):
    axes[1].scatter(x,y,s=550,c='#2b788c' if i in grupo else '#bf573f',zorder=3)
    axes[1].text(x,y,rotulos[i],ha='center',va='center',color='white',zorder=4)
axes[1].set_title('Grafo de correspondências verificadas')
axes[1].set_aspect('equal'); axes[1].axis('off')
fig.tight_layout()
figura(fig,'04_conectividade')
rejeitadas = [i for i in range(len(quadros)) if i not in grupo]
print('Referência:',rotulos[referencia])
print('Sequência inicial:', ' → '.join(rotulos[i] for i in ordem))
print('Fora do componente:', ', '.join(rotulos[i] for i in rejeitadas) or 'nenhuma')
painel([quadros[i].rgb for i in ordem], [rotulos[i] for i in ordem], '04_ordem', colunas=len(ordem), altura=3.3)

Na coleta fornecida, escolhemos SIFT pelo menor erro médio nos inliers (1,13 contra 1,30 pixels) e pela aceitação de nove arestas, contra oito com ORB. ORB é mais rápido, produz mais inliers no total e tem maior taxa ponderada. Essas medidas avaliam aspectos diferentes e usam conjuntos de correspondências distintos; não demonstram superioridade geral de um detector. Os tempos apresentados são de extração, não do pipeline completo.

A conexão mais fraca reaparece com `τ = 0,80`; reduzir para 0,75 desconecta uma vista válida. O teste com 0,85 mostra o custo de relaxar o filtro. A escolha de 0,80 usa este conjunto de desenvolvimento e não demonstra generalização a outras coletas.

A aceitação das arestas não é monotônica em `τ`: além do ratio test, depende da hipótese escolhida pelo RANSAC e das verificações de plausibilidade da homografia.

## 5. Homografia, RANSAC e alinhamento

Para pontos homogêneos, adotamos \(\mathbf{x}_j\sim H_{ij}\mathbf{x}_i\). A matriz tem oito graus de liberdade, pois uma escala comum não altera a projeção. Quatro pares em configuração não degenerada determinam uma solução; pontos colineares não bastam [R4; R6, §§2 e 4].

O **DLT normalizado** centraliza os pontos, ajusta sua distância média à origem para \(\sqrt{2}\), monta \(A\mathbf{h}=0\) e obtém o vetor singular associado ao menor valor singular. Depois desfaz a normalização. O exemplo usa dados sintéticos de geometria conhecida; nas fotografias, precisamos tratar outliers.

In [ ]:
def dlt_normalizado(origem, destino):
    origem, destino = np.asarray(origem, float), np.asarray(destino, float)
    if origem.shape != destino.shape or origem.ndim != 2 or origem.shape[1] != 2 or len(origem) < 4:
        raise ValueError('São necessárias ao menos quatro correspondências 2D.')
    def normalizar(p):
        centro = p.mean(axis=0)
        if np.linalg.matrix_rank(p-centro) < 2:
            raise ValueError('Pontos colineares não definem uma homografia.')
        escala = np.sqrt(2) / np.mean(np.linalg.norm(p-centro, axis=1))
        T = np.array([[escala, 0, -escala*centro[0]], [0, escala, -escala*centro[1]], [0, 0, 1.]])
        return projetar(T, p), T
    a, Ta = normalizar(origem)
    b, Tb = normalizar(destino)
    linhas = []
    for (x, y), (u, v) in zip(a, b):
        linhas.extend([[-x, -y, -1, 0, 0, 0, u*x, u*y, u],
                       [0, 0, 0, -x, -y, -1, v*x, v*y, v]])
    A = np.array(linhas)
    if np.linalg.matrix_rank(A) < 8:
        raise ValueError('Configuração degenerada.')
    _, _, vt = np.linalg.svd(A, full_matrices=True)
    H = np.linalg.inv(Tb) @ vt[-1].reshape(3, 3) @ Ta
    return H / H[2, 2]


def transformar_plano(quadros, transforms, grupo, limite=30_000_000):
    for i in grupo:
        if not homografia_plausivel(transforms[i], quadros[i].rgb):
            raise ValueError('Uma única projeção plana é inadequada para esta amplitude.')
    bordas = np.concatenate([projetar(transforms[i], cantos(quadros[i].rgb)) for i in grupo])
    minimo = np.floor(bordas.min(axis=0)).astype(int)
    maximo = np.ceil(bordas.max(axis=0)).astype(int)
    w, h = (maximo - minimo + 1).tolist()
    if w*h > limite or max(w, h) > 30000:
        raise ValueError(f'Canvas plano excessivo: {w} x {h}.')
    T = np.array([[1., 0, -minimo[0]], [0, 1., -minimo[1]], [0, 0, 1.]])
    imagens, mascaras = [], []
    for i in grupo:
        rgb = quadros[i].rgb
        H = T @ transforms[i]
        imagens.append(cv.warpPerspective(rgb.astype(np.float32)/255, H, (w, h)))
        mascaras.append(cv.warpPerspective(np.ones(rgb.shape[:2], np.uint8), H, (w, h), flags=cv.INTER_NEAREST).astype(bool))
    return imagens, mascaras, T

In [ ]:
pontos_teste = np.array([[0,0],[400,0],[400,300],[0,300],[200,120],[80,250]],float)
H_teste = np.array([[1.1,.06,40],[-.02,.95,12],[.0002,-.0001,1.]])
alvos_teste = projetar(H_teste,pontos_teste)
H_dlt = dlt_normalizado(pontos_teste,alvos_teste)
print('H recuperada pelo DLT:\n', np.round(H_dlt,6))
print('Erro máximo no exemplo:',np.linalg.norm(projetar(H_dlt,pontos_teste)-alvos_teste,axis=1).max())
plano_imgs, plano_masks, T_plano = transformar_plano(quadros,
    {par_exemplo.i:par_exemplo.H, par_exemplo.j:np.eye(3)}, [par_exemplo.i,par_exemplo.j])
painel([plano_imgs[0],plano_imgs[1],
        (plano_imgs[0]*plano_masks[0][...,None]+plano_imgs[1]*plano_masks[1][...,None])/
        np.maximum((plano_masks[0].astype(float)+plano_masks[1])[...,None],1)],
       ['Origem transformada','Destino como referência','Sobreposição no plano'],
       '05_homografia',colunas=3,altura=5)
del plano_imgs, plano_masks

**RANSAC** estima modelos a partir de amostras pequenas, conta correspondências compatíveis e refina a melhor hipótese [R3]. Aqui, `findHomography` usa tolerância de 3 pixels na resolução de trabalho, confiança de 0,999 e até 10 mil iterações. A semente deriva do par de pixels e torna o resultado repetível. Não implementamos novamente a rotina robusta do OpenCV.

O erro por correspondência é \(e_k=\|\pi(H_{ij}\mathbf{x}_{ik})-\mathbf{x}_{jk}\|_2\). A taxa de inliers usa como denominador os matches **após** o ratio test. A tabela contém todas as arestas utilizadas pelo ajuste; a coluna `arvore` identifica as seis conexões que inicializam as transformações. Média, mediana e percentil 90 são calculados somente nos inliers.

`warpPerspective` transforma a imagem e uma máscara de validade separada. Assim, preto não significa pixel ausente. Cada imagem é reamostrada a partir da fonte, evitando transformar repetidamente o mosaico já interpolado.

In [ ]:
def estatisticas_pares(pares):
    linhas = []
    for p in pares:
        e = p.erros[p.inliers]
        linhas.append({'origem':f'I{p.i+1:02}', 'destino':f'I{p.j+1:02}',
                       'matches':len(p.inliers), 'inliers':p.n, 'taxa':round(p.taxa,4),
                       'erro_medio_px':float(np.mean(e)) if len(e) else None,
                       'erro_mediano_px':float(np.median(e)) if len(e) else None,
                       'erro_p90_px':float(np.percentile(e,90)) if len(e) else None,
                       'cobertura':p.cobertura, 'aceito':p.aceito})
    return linhas


def salvar_csv(caminho, linhas):
    if not linhas:
        return
    with Path(caminho).open('w', newline='', encoding='utf-8') as arquivo:
        writer = csv.DictWriter(arquivo, fieldnames=list(linhas[0]))
        writer.writeheader()
        writer.writerows(linhas)

In [ ]:
medidas_pares = estatisticas_pares(pares)
conexoes_arvore = {tuple(sorted(a)) for a in arestas}
utilizados = []
for p,linha in zip(pares,medidas_pares):
    if p.aceito:
        utilizados.append({**linha, 'arvore':(p.i,p.j) in conexoes_arvore})
tabela(utilizados, ['origem','destino','matches','inliers','taxa','erro_medio_px','erro_mediano_px','erro_p90_px','arvore'])
fig,ax=plt.subplots(figsize=(10,4))
aceitos=[p for p in pares if p.aceito]
ax.boxplot([p.erros[p.inliers] for p in aceitos],tick_labels=[f'{rotulos[p.i]}–{rotulos[p.j]}' for p in aceitos],showfliers=False)
ax.axhline(cfg.ransac_px,color='#bf573f',ls='--',label='Tolerância do RANSAC')
ax.set(ylabel='Erro de reprojeção nos inliers (px)',xlabel='Arestas aceitas')
ax.tick_params(axis='x',rotation=35); ax.legend()
fig.tight_layout(); figura(fig,'05_residuos')

### Ajuste global e projeção cilíndrica

A varredura desta coleta é ampla. Encadear todas as vistas num único plano aproxima uma extremidade do infinito projetivo e produz um canvas inadequado. Mantemos a demonstração plana entre duas vistas e usamos uma projeção cilíndrica para o panorama completo [R5; R6, §§2.3 e 5].

Assumimos uma câmera com distância focal comum, centro principal no centro da imagem e rotação predominante. A inicialização usa \(H_{ij}\approx K_jR_j^{-1}R_iK_i^{-1}\). Ajustamos a focal e três parâmetros de rotação por vista, fixando a referência. Todas as arestas aceitas contribuem, com até 180 inliers distribuídos por par e perda robusta `soft_l1`.

O objetivo aproxima raios correspondentes no mesmo referencial. O RMS abaixo mede componentes da diferença entre raios unitários, multiplicada pela focal inicial: **pixels equivalentes**, distintos do erro de reprojeção da tabela anterior. O modelo não inclui translação, profundidade ou distorção de lente; paralaxe e reflexos podem persistir.

In [ ]:
def intrinseca(quadro, focal):
    h, w = quadro.rgb.shape[:2]
    return np.array([[focal, 0, w/2], [0, focal, h/2], [0, 0, 1.]], float)


def rotacao_proxima(M):
    if np.linalg.det(M) < 0:
        M = -M
    u, _, vt = np.linalg.svd(M)
    return u @ np.diag([1, 1, np.linalg.det(u @ vt)]) @ vt


def ajustar_cameras(quadros, pares, grupo, referencia, transforms):
    validos = [p for p in pares if p.aceito and p.i in grupo and p.j in grupo]
    tamanho = float(max(quadros[referencia].rgb.shape[:2]))
    def erro_focal(x):
        focal = np.exp(x[0])
        residuos = []
        for p in validos:
            R = np.linalg.inv(intrinseca(quadros[p.j], focal)) @ p.H @ intrinseca(quadros[p.i], focal)
            R /= np.cbrt(np.linalg.det(R))
            residuos.extend((R.T @ R - np.eye(3)).ravel())
        return np.array(residuos)
    f0 = least_squares(erro_focal, [np.log(tamanho)], bounds=([np.log(.35*tamanho)], [np.log(3*tamanho)]), loss='soft_l1')
    focal0 = float(np.exp(f0.x[0]))
    livres = [i for i in grupo if i != referencia]
    Kref = intrinseca(quadros[referencia], focal0)
    rotacoes0 = {i: rotacao_proxima(np.linalg.inv(Kref) @ transforms[i] @ intrinseca(quadros[i], focal0)) for i in grupo}
    x0 = np.r_[np.log(focal0), np.concatenate([Rotation.from_matrix(rotacoes0[i]).as_rotvec() for i in livres])]
    observacoes = []
    for p in validos:
        ids = np.flatnonzero(p.inliers)
        ids = ids[np.linspace(0, len(ids)-1, min(len(ids), 180)).astype(int)]
        observacoes.append((p.i, p.j, p.origem[ids], p.destino[ids]))
    def desempacotar(x):
        Rs = {referencia: np.eye(3)}
        Rs.update({i: Rotation.from_rotvec(x[1+3*k:4+3*k]).as_matrix() for k, i in enumerate(livres)})
        return float(np.exp(x[0])), Rs
    def residuos(x):
        focal, Rs = desempacotar(x)
        resultado = []
        for i, j, a, b in observacoes:
            raios = []
            for idx, p in [(i, a), (j, b)]:
                v = np.c_[p, np.ones(len(p))] @ np.linalg.inv(intrinseca(quadros[idx], focal)).T
                v /= np.linalg.norm(v, axis=1, keepdims=True)
                raios.append(v @ Rs[idx].T)
            resultado.append((focal0 * (raios[0] - raios[1])).ravel())
        return np.concatenate(resultado)
    lower = np.r_[np.log(.35*tamanho), np.full(len(x0)-1, -np.inf)]
    upper = np.r_[np.log(3*tamanho), np.full(len(x0)-1, np.inf)]
    fit = least_squares(residuos, x0, bounds=(lower, upper), loss='soft_l1', f_scale=2.0, max_nfev=100)
    focal, Rs = desempacotar(fit.x)
    resumo = {'focal_inicial_px': focal0, 'focal_px': focal,
              'residuo_raios_antes_px': float(np.sqrt(np.mean(residuos(x0)**2))),
              'residuo_raios_depois_px': float(np.sqrt(np.mean(residuos(fit.x)**2))),
              'convergiu': bool(fit.success), 'avaliacoes': int(fit.nfev)}
    return focal, Rs, resumo

In [ ]:
def srgb_linear(rgb):
    return np.where(rgb <= .04045, rgb/12.92, ((rgb+.055)/1.055)**2.4).astype(np.float32)


def linear_srgb(rgb):
    rgb = np.maximum(rgb, 0)
    return np.clip(np.where(rgb <= .0031308, 12.92*rgb, 1.055*rgb**(1/2.4)-.055), 0, 1)


def transformar_cilindro(quadros, grupo, focal, rotacoes):
    warper = cv.PyRotationWarper('cylindrical', focal)
    partes, validades, offsets = [], [], []
    for i in grupo:
        q = quadros[i]
        K = intrinseca(q, focal).astype(np.float32)
        R = rotacoes[i].astype(np.float32)
        offset, img = warper.warp(srgb_linear(q.rgb.astype(np.float32)/255), K, R, cv.INTER_LINEAR, cv.BORDER_REFLECT)
        _, mask = warper.warp(np.ones(q.rgb.shape[:2], np.uint8), K, R, cv.INTER_NEAREST, cv.BORDER_CONSTANT)
        offsets.append(offset)
        partes.append(img)
        validades.append(mask.astype(bool))
    canto = np.min(offsets, axis=0)
    fim = np.max([np.array(off)+[im.shape[1], im.shape[0]] for off, im in zip(offsets, partes)], axis=0)
    w, h = (fim - canto).astype(int)
    if w*h > 30_000_000:
        raise ValueError('Canvas cilíndrico excede o limite de memória previsto.')
    imagens, mascaras = [], []
    for off, img, mask in zip(offsets, partes, validades):
        x, y = (np.array(off)-canto).astype(int)
        out = np.zeros((h,w,3), np.float32)
        valid = np.zeros((h,w), bool)
        out[y:y+img.shape[0], x:x+img.shape[1]] = img
        valid[y:y+img.shape[0], x:x+img.shape[1]] = mask
        imagens.append(out)
        mascaras.append(valid)
    centros = {i: float(np.arctan2(rotacoes[i][0,2], rotacoes[i][2,2])) for i in grupo}
    ordem = sorted(grupo, key=lambda i: centros[i])
    return imagens, mascaras, ordem, canto

In [ ]:
try:
    ensaio_plano = transformar_plano(quadros,H,grupo)
    diagnostico_plano = 'O canvas plano passou nas verificações de tamanho e geometria.'
    del ensaio_plano
except ValueError as erro:
    diagnostico_plano = str(erro)
print(diagnostico_plano)
focal, rotacoes, ajuste = ajustar_cameras(quadros,pares,grupo,referencia,H)
imagens, mascaras, ordem, deslocamento = transformar_cilindro(quadros,grupo,focal,rotacoes)
if any(W[i,j]==0 for i,j in zip(ordem,ordem[1:])):
    raise ValueError('A sequência cilíndrica não é sustentada pelo grafo.')
tabela([ajuste])
print('Sequência final:', ' → '.join(rotulos[i] for i in ordem))

O cilindro representa um raio \((X,Y,Z)\) por \(u=f\,\operatorname{atan2}(X,Z)\) e \(v=fY/\sqrt{X^2+Z^2}\). `PyRotationWarper` realiza essa projeção e a interpolação. Imagens e máscaras compartilham os mesmos parâmetros. A sequência abaixo mostra a incorporação progressiva das vistas na geometria já ajustada, com média simples apenas para visualizar sobreposições.

In [ ]:
def media_valida(imagens, mascaras):
    total = np.zeros_like(imagens[0], dtype=np.float32)
    peso = np.zeros(imagens[0].shape[:2], np.float32)
    for img, mask in zip(imagens, mascaras):
        total += img * mask[...,None]
        peso += mask
    return total / np.maximum(peso[...,None], 1)


def feathering(imagens, mascaras):
    total = np.zeros_like(imagens[0], dtype=np.float32)
    peso = np.zeros(imagens[0].shape[:2], np.float32)
    for img, mask in zip(imagens, mascaras):
        padded = np.pad(mask.astype(np.uint8), 1)
        d = cv.distanceTransform(padded, cv.DIST_L2, 3)[1:-1,1:-1]
        total += img*d[...,None]
        peso += d
    return total / np.maximum(peso[...,None], 1e-8)


def maior_retangulo(mask, passo=4):
    h, w = mask.shape
    passo = min(passo, max(1, min(h,w)//8))
    hh, ww = h//passo, w//passo
    reduzida = mask[:hh*passo,:ww*passo].reshape(hh,passo,ww,passo).all(axis=(1,3))
    alturas = np.zeros(ww, int)
    melhor = (0, (0,0,0,0))
    for y, linha in enumerate(reduzida):
        alturas = np.where(linha, alturas+1, 0)
        pilha = []
        for x in range(ww+1):
            altura = int(alturas[x]) if x < ww else 0
            inicio = x
            while pilha and pilha[-1][1] > altura:
                esquerda, valor = pilha.pop()
                area = valor*(x-esquerda)
                if area > melhor[0]:
                    melhor = (area, (esquerda, y-valor+1, x, y+1))
                inicio = esquerda
            if not pilha or pilha[-1][1] < altura:
                pilha.append((inicio,altura))
    if melhor[0] == 0:
        raise ValueError('Não há região válida para recorte.')
    return tuple(int(x*passo) for x in melhor[1])


def recortar(img, caixa):
    x0,y0,x1,y1 = caixa
    return img[y0:y1,x0:x1]

In [ ]:
caixa = maior_retangulo(np.logical_or.reduce(mascaras))
progresso=[]; titulos=[]
for n in sorted(set([1,2,min(4,len(grupo)),len(grupo)])):
    ids=[grupo.index(i) for i in ordem[:n]]
    progresso.append(linear_srgb(recortar(media_valida([imagens[i] for i in ids],[mascaras[i] for i in ids]),caixa)))
    titulos.append(f'{n} vista(s)')
painel(progresso,titulos,'05_progressao',colunas=2,altura=6)
del progresso

## 6. Composição e remoção de fantasmas

A média de pixels válidos mistura objetos que ocupam posições diferentes. O **feathering** pondera cada imagem pela distância à sua borda, mas ainda mistura fontes. Uma **costura** escolhe qual imagem fornece cada região; o blending posterior suaviza a transição [R6, §6; R9].

Primeiro aproximamos o JPEG como sRGB e convertemos para intensidade linear. Ganhos RGB são estimados nas sobreposições por razões medianas, excluindo pixels muito escuros ou saturados. Fixamos a referência e limitamos os ganhos a [0,5; 2]. Isso é uma compensação global de exposição e cor, inspirada em [R5, §6], não uma calibração radiométrica da câmera.

A função de transferência usa a conversão por trechos de [sRGB e intensidade linear descrita pelo W3C](https://www.w3.org/TR/2026/CRD-css-color-4-20260908/#color-conversion-code).

In [ ]:
def compensar_exposicao(imagens, mascaras, referencia):
    n = len(imagens)
    equacoes, valores = [], []
    for i, j in itertools.combinations(range(n), 2):
        a, b = imagens[i][::4,::4], imagens[j][::4,::4]
        valid = mascaras[i][::4,::4] & mascaras[j][::4,::4]
        valid &= (a.min(axis=2) > .015) & (b.min(axis=2) > .015)
        valid &= (a.max(axis=2) < .8) & (b.max(axis=2) < .8)
        if valid.sum() < 100:
            continue
        linha = np.zeros(n)
        linha[i], linha[j] = 1, -1
        equacoes.append(linha)
        valores.append(np.median(np.log(b[valid]) - np.log(a[valid]), axis=0))
    if not equacoes:
        return imagens, np.ones((n,3))
    ancora = np.zeros(n)
    ancora[referencia] = 1
    equacoes.append(ancora)
    valores.append(np.zeros(3))
    logs = np.linalg.lstsq(np.array(equacoes), np.array(valores), rcond=None)[0]
    ganhos = np.clip(np.exp(logs), .5, 2.)
    return [np.clip(im*ganho, 0, 1).astype(np.float32) for im, ganho in zip(imagens, ganhos)], ganhos

In [ ]:
corrigidas, ganhos = compensar_exposicao(imagens,mascaras,grupo.index(referencia))
tabela([{'imagem':rotulos[i],'ganho_R':g[0],'ganho_G':g[1],'ganho_B':g[2]} for i,g in zip(grupo,ganhos)])
painel([linear_srgb(recortar(media_valida(imagens,mascaras),caixa)),
        linear_srgb(recortar(media_valida(corrigidas,mascaras),caixa))],
       ['Média sem compensação','Média com compensação'], '06_exposicao',colunas=1,altura=7)

### Costura por programação dinâmica

Na sobreposição, o custo é a média quadrática da diferença RGB linear. Fora dela, o custo é infinito. Para uma costura vertical, a recorrência é

\[
C(y,x)=E(y,x)+\min_{d\in\{-1,0,1\}} C(y-1,x+d).
\]

O caminho termina no menor custo da última linha e é recuperado pelos predecessores. Adaptamos a fronteira de menor erro de *Image Quilting* [R7, §2.1], originalmente proposta para síntese de textura. A nova vista fornece o lado direito da costura, seguindo a ordem horizontal inferida.

Atualizamos o mapa de fontes sequencialmente. Isso não garante um ótimo conjunto de costuras para todas as imagens. Se a sobreposição não admitir um caminho vertical conectado, o código usa o corte em grafo do OpenCV, relacionado à seleção de fontes por graph cuts [R10]. O registro informa qual ramo foi utilizado.

In [ ]:
def costura_minima(custo):
    custo = np.asarray(custo, float)
    if custo.ndim != 2 or min(custo.shape) == 0 or np.isnan(custo).any():
        raise ValueError('Mapa de custo inválido.')
    h, w = custo.shape
    acumulado = custo.copy()
    pais = np.zeros((h,w), np.int8)
    for y in range(1,h):
        anterior = np.pad(acumulado[y-1], (1,1), constant_values=np.inf)
        escolhas = np.stack([anterior[:w], anterior[1:w+1], anterior[2:w+2]])
        direcao = escolhas.argmin(axis=0)
        acumulado[y] += escolhas[direcao, np.arange(w)]
        pais[y] = direcao - 1
    x = int(acumulado[-1].argmin())
    if not np.isfinite(acumulado[-1,x]):
        raise ValueError('Não há caminho conectado na sobreposição.')
    caminho = np.empty(h, int)
    caminho[-1] = x
    for y in range(h-1,0,-1):
        caminho[y-1] = caminho[y] + pais[y,caminho[y]]
    return caminho, acumulado


def escolher_fontes(imagens, mascaras, ordem_local):
    fontes = np.full(mascaras[0].shape, -1, np.int16)
    composto = np.zeros_like(imagens[0])
    uniao = np.zeros_like(mascaras[0])
    registros = []
    for i in ordem_local:
        img, mask = imagens[i], mascaras[i]
        overlap = uniao & mask
        nova = mask & ~uniao
        metodo = 'sem_sobreposicao'
        if overlap.any():
            yy, xx = np.where(overlap)
            y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1
            diferenca = np.mean((composto[y0:y1,x0:x1] - img[y0:y1,x0:x1])**2, axis=2)
            custo = np.where(overlap[y0:y1,x0:x1], diferenca, np.inf)
            try:
                caminho, _ = costura_minima(custo)
                lado = np.arange(x1-x0)[None,:] > caminho[:,None]
                nova[y0:y1,x0:x1] |= lado & overlap[y0:y1,x0:x1]
                metodo = 'programacao_dinamica'
            except ValueError:
                a, b = cv.UMat(uniao.astype(np.uint8)*255), cv.UMat(mask.astype(np.uint8)*255)
                cv.detail_GraphCutSeamFinder('COST_COLOR_GRAD').find([composto*255, img*255], [(0,0),(0,0)], [a,b])
                nova |= (b.get() > 0) & overlap
                metodo = 'corte_em_grafo'
            registros.append({'imagem': int(i), 'metodo': metodo, 'sobreposicao_px': int(overlap.sum())})
        composto[nova] = img[nova]
        fontes[nova] = i
        uniao |= mask
    selecoes = [fontes == i for i in range(len(imagens))]
    return selecoes, fontes, registros

In [ ]:
selecoes, fontes, costuras = escolher_fontes(corrigidas,mascaras,[grupo.index(i) for i in ordem])
tabela(costuras)
a,b=[grupo.index(i) for i in ordem[:2]]
overlap=mascaras[a]&mascaras[b]
yy,xx=np.where(overlap)
y0,y1,x0,x1=yy.min(),yy.max()+1,xx.min(),xx.max()+1
custo=np.where(overlap[y0:y1,x0:x1],
    np.mean((corrigidas[a][y0:y1,x0:x1]-corrigidas[b][y0:y1,x0:x1])**2,axis=2),np.inf)
fig,axes=plt.subplots(1,3,figsize=(12,5))
limite=np.percentile(custo[np.isfinite(custo)],95)
axes[0].imshow(np.ma.masked_invalid(custo),cmap='magma',vmax=limite)
axes[0].set_title('Diferença na sobreposição')
try:
    caminho,acumulado=costura_minima(custo)
    axes[1].imshow(np.ma.masked_invalid(acumulado),cmap='viridis')
    axes[1].plot(caminho,np.arange(len(caminho)),color='#ffce55',lw=1)
    axes[1].set_title('Custo acumulado e caminho')
except ValueError:
    axes[1].text(.1,.5,'Esta sobreposição exige corte em grafo.',wrap=True)
axes[2].imshow(fontes[y0:y1,x0:x1],cmap='tab10',vmin=0,vmax=9)
axes[2].set_title('Fonte escolhida no mosaico')
for ax in axes: ax.axis('off')
fig.tight_layout(); figura(fig,'06_costura')

### Blending multibanda

Uma pirâmide laplaciana separa frequências da imagem; uma pirâmide gaussiana suaviza a máscara em escalas correspondentes. Combinamos cada banda com pesos normalizados e reconstruímos a imagem [R8]. A seleção da costura orienta os pesos, mas as bandas mais grossas ainda misturam vizinhanças.

Usamos cinco níveis e preservamos tamanhos ímpares na reconstrução. A comparação mantém geometria, compensação de exposição e recorte iguais. A variante “costura” mostra a seleção direta; a final acrescenta o blending multibanda.

In [ ]:
def piramide_laplaciana(img, niveis):
    gaussianas = [np.asarray(img, np.float32)]
    for _ in range(niveis-1):
        if min(gaussianas[-1].shape[:2]) < 3:
            break
        gaussianas.append(cv.pyrDown(gaussianas[-1]))
    laplacianas = []
    for a, b in zip(gaussianas, gaussianas[1:]):
        laplacianas.append(a - cv.pyrUp(b, dstsize=(a.shape[1],a.shape[0])))
    return laplacianas + [gaussianas[-1]]


def reconstruir(piramide):
    img = piramide[-1].copy()
    for nivel in piramide[-2::-1]:
        img = cv.pyrUp(img, dstsize=(nivel.shape[1], nivel.shape[0])) + nivel
    return img


def multibanda(imagens, selecoes, niveis):
    somas, pesos = None, None
    for img, mask in zip(imagens, selecoes):
        lap = piramide_laplaciana(img, niveis)
        gauss = [mask.astype(np.float32)]
        for _ in lap[1:]:
            gauss.append(cv.pyrDown(gauss[-1]))
        if somas is None:
            somas = [np.zeros_like(a) for a in lap]
            pesos = [np.zeros_like(g) for g in gauss]
        for k, (a,g) in enumerate(zip(lap, gauss)):
            somas[k] += a * g[...,None]
            pesos[k] += g
    resultado = reconstruir([a/np.maximum(w[...,None], 1e-8) for a,w in zip(somas,pesos)])
    resultado[~np.logical_or.reduce(selecoes)] = 0
    return np.clip(resultado, 0, 1)

In [ ]:
exemplo_piramide = srgb_linear(cv.resize(quadros[par_exemplo.i].rgb,(300,400)).astype(np.float32)/255)
bandas = piramide_laplaciana(exemplo_piramide,cfg.niveis)
painel([linear_srgb(exemplo_piramide)]+[np.clip(b*4+.5,0,1) for b in bandas[:-1]]+[linear_srgb(bandas[-1])],
       ['Imagem']+[f'Banda {i+1} · contraste ×4' for i in range(len(bandas)-1)]+['Baixas frequências'],
       '06_piramides',colunas=3,altura=7)
variantes = {'Média':media_valida(corrigidas,mascaras),
             'Feathering':feathering(corrigidas,mascaras),
             'Costura':media_valida(corrigidas,selecoes),
             'Multibanda':multibanda(corrigidas,selecoes,cfg.niveis)}
final = variantes['Multibanda']
painel([linear_srgb(recortar(v,caixa)) for v in variantes.values()],list(variantes),
       '06_composicao',colunas=1,altura=13)
Image.fromarray(np.round(linear_srgb(recortar(final,caixa))*255).astype(np.uint8)).save(SAIDA/'panorama.png')
Image.fromarray(np.round(linear_srgb(recortar(final,caixa))*255).astype(np.uint8)).save(SAIDA/'panorama.jpg',quality=94)
print('Panorama:',tuple(reversed(recortar(final,caixa).shape[:2])),'pixels')

### Região com movimento: as mesmas coordenadas em todas as variantes

Na coleta original, a região do carro foi anotada **somente para a inspeção**, no ponto (414, 1078) da vista de hash `4a107663e018`, à resolução de 1600 pixels. Essa anotação não participa de correspondências, ordem, ajuste ou costura. A projeção leva o ponto ao canvas; todas as variantes usam a mesma janela de 300 × 160 pixels.

As duas fontes alinhadas permitem verificar a mudança real na cena. Um mapa de diferenças, sozinho, também responde a exposição e paralaxe e não identifica movimento com certeza [R9]. Não calculamos PSNR ou SSIM contra uma “verdade” inexistente. Como medida auxiliar, calculamos a distância à fonte disponível mais próxima por pixel na região: valores menores indicam menos mistura, mas não garantem que o objeto certo tenha sido preservado.

In [ ]:
roi_medidas=[]
uid_carro='4a107663e018'
indices_carro=[i for i,q in enumerate(quadros) if q.uid==uid_carro and i in grupo]
if indices_carro:
    i=indices_carro[0]
    warper=cv.PyRotationWarper('cylindrical',focal)
    centro=np.array(warper.warpPoint((414.,1078.),intrinseca(quadros[i],focal).astype(np.float32),rotacoes[i].astype(np.float32)))-deslocamento
    cx,cy=np.round(centro).astype(int)
    roi=(max(0,cx-150),max(0,cy-80),min(final.shape[1],cx+150),min(final.shape[0],cy+80))
    presentes=[k for k,m in enumerate(mascaras) if recortar(m,roi).mean()>.95]
    vistas=[linear_srgb(recortar(corrigidas[k],roi)) for k in presentes]
    legendas=[f'Fonte {rotulos[grupo[k]]}' for k in presentes]
    painel(vistas+[linear_srgb(recortar(v,roi)) for v in variantes.values()],
           legendas+list(variantes),'06_carro',colunas=2,altura=7)
    comum=np.logical_and.reduce([recortar(mascaras[k],roi) for k in presentes])
    for nome,v in variantes.items():
        distancias=np.stack([np.mean(np.abs(recortar(v,roi)-recortar(corrigidas[k],roi)),axis=2) for k in presentes])
        roi_medidas.append({'variante':nome,'distancia_fonte_linear':float(distancias.min(axis=0)[comum].mean()),
                            'pixels_avaliados':int(comum.sum())})
    tabela(roi_medidas)
else:
    print('A anotação do carro pertence à coleta original; a montagem desta pasta segue sem essa figura.')

A costura evita a transparência produzida pela média na região do carro. A comparação deve ser lida junto das fontes, pois preservar uma ocorrência ou escolher o fundo são soluções possíveis. O multibanda suaviza transições, mas não substitui a seleção de fontes.

No panorama completo ainda há desalinhamentos em fios, contornos próximos e detalhes da fachada. A câmera observou objetos em profundidades diferentes; rotação pura e uma única focal são aproximações. Luzes saturadas e reflexos da lente também permanecem. O recorte retangular remove bordas sem amostras, ao custo de perder parte do céu e da sacada da vista mais alta. A versão com cobertura completa abaixo preserva a extensão capturada.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].imshow(linear_srgb(final)); axes[0].set_title('Canvas completo, sem recorte')
x0,y0,x1,y1=caixa
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,color='#ffce55',lw=1.5))
axes[1].imshow(np.sum(mascaras,axis=0),cmap='viridis',vmin=0,vmax=len(grupo))
axes[1].set_title('Número de fontes por pixel')
for ax in axes: ax.axis('off')
fig.tight_layout(); figura(fig,'06_cobertura')

### Comparação com o montador pronto do OpenCV

`Stitcher` oferece uma referência prática [documentação oficial](https://docs.opencv.org/4.x/d8/d19/tutorial_stitcher.html). Ele recebe apenas as imagens do componente selecionado, na mesma resolução de entrada. Seus parâmetros internos e sua projeção podem diferir; a comparação visual não isola um único fator. Um código de falha também é um resultado e será registrado, sem substituir a implementação anterior.

In [ ]:
cv.setRNGSeed(cfg.seed)
stitcher=cv.Stitcher_create(cv.Stitcher_PANORAMA)
inicio=time.perf_counter()
status, pano_cv=stitcher.stitch([cv.cvtColor(quadros[i].rgb,cv.COLOR_RGB2BGR) for i in grupo])
comparacao_opencv={'status':int(status),'tempo_s':time.perf_counter()-inicio}
print(comparacao_opencv)
if status==cv.Stitcher_OK:
    pano_cv=cv.cvtColor(pano_cv,cv.COLOR_BGR2RGB)
    Image.fromarray(pano_cv).save(SAIDA/'panorama_opencv.jpg',quality=92)
    painel([linear_srgb(recortar(final,caixa)),pano_cv],['Implementação do notebook','OpenCV Stitcher'],
           '06_opencv',colunas=1,altura=7)

## Execução automática e registro

A função abaixo reúne as etapas de montagem. Ela usa somente imagens decodificadas, correspondências e parâmetros; o executor de linha de comando executa o próprio notebook. As células marcadas `definicoes` também são carregadas pelos testes, mantendo uma única implementação dos algoritmos.

O ensaio de invariância em `tools/validar_execucao.py` cria cinco pastas temporárias com nomes opacos, ordem de criação aleatória e PNGs sem EXIF. Os pixels de trabalho são preservados. A detecção, o matching, o ajuste e a composição são recalculados em cada pasta e comparados à referência. Isso verifica a independência de nomes e listagem para esta coleta, não a robustez a toda possível cena.

In [ ]:
def montar(quadros, pares, cfg):
    W, grupo, ref, H, arestas, ordem = alinhar_grafo(quadros, pares)
    focal, Rs, ajuste = ajustar_cameras(quadros, pares, grupo, ref, H)
    imagens, mascaras, ordem, deslocamento = transformar_cilindro(quadros, grupo, focal, Rs)
    if any(W[i,j] == 0 for i,j in zip(ordem,ordem[1:])):
        raise ValueError('A sequência cilíndrica não é sustentada pelo grafo.')
    corrigidas, ganhos = compensar_exposicao(imagens, mascaras, grupo.index(ref))
    selecoes, fontes, costuras = escolher_fontes(corrigidas, mascaras, [grupo.index(i) for i in ordem])
    final = multibanda(corrigidas, selecoes, cfg.niveis)
    caixa = maior_retangulo(np.logical_or.reduce(mascaras))
    return {'W':W, 'grupo':grupo, 'referencia':ref, 'H':H, 'arestas':arestas,
            'ordem':ordem, 'focal':focal, 'rotacoes':Rs, 'ajuste':ajuste,
            'imagens':imagens, 'mascaras':mascaras, 'corrigidas':corrigidas,
            'ganhos':ganhos, 'selecoes':selecoes, 'fontes':fontes, 'costuras':costuras,
            'final':final, 'caixa':caixa, 'deslocamento':deslocamento}

In [ ]:
versoes={nome:importlib.metadata.version(nome) for nome in ['numpy','opencv-python-headless','scipy','matplotlib','Pillow','tifffile','nbformat','nbclient']}
metricas={'configuracao':asdict(cfg),'versoes':versoes,'python':platform.python_version(),
    'inventario':inventario,'comparacao_detectores':comparacao,'sensibilidade_ratio':sensibilidade,
    'matriz_inliers':W,'componente':[rotulos[i] for i in grupo],
    'rejeitadas':[rotulos[i] for i in rejeitadas],'ordem':[rotulos[i] for i in ordem],
    'ordem_uids':[quadros[i].uid for i in ordem],'referencia':rotulos[referencia],
    'arestas_arvore':[[rotulos[i],rotulos[j]] for i,j in arestas],
    'pares_sift':medidas_pares,'ajuste_global':ajuste,'ganhos_rgb':ganhos,
    'costuras':costuras,'canvas_hw':final.shape[:2],'recorte_xyxy':caixa,
    'panorama_wh':tuple(reversed(recortar(final,caixa).shape[:2])),
    'roi_carro':roi_medidas,'opencv':comparacao_opencv,'diagnostico_plano':diagnostico_plano}
(SAIDA/'metricas.json').write_text(json.dumps(metricas,ensure_ascii=False,indent=2,default=serializavel))
salvar_csv(SAIDA/'pares_sift.csv',medidas_pares)
salvar_csv(SAIDA/'pares_orb.csv',estatisticas_pares(pares_por_metodo['ORB']))
salvar_csv(SAIDA/'detectores.csv',comparacao)
salvar_csv(SAIDA/'caracteristicas.csv',medidas_features)
print('Resultados gravados em',SAIDA.resolve())
validacao=SAIDA/'validacao.json'
if validacao.exists():
    tabela(json.loads(validacao.read_text())['embaralhamentos'])
else:
    print('Execute tools/validar_execucao.py para registrar os cinco embaralhamentos.')

## Discussão

O conjunto permite demonstrar as seis etapas: características locais, filtragem de correspondências, reconhecimento do componente, geometria robusta e composição. A foto com enquadramento alto permanece conectada, enquanto a fotografia de outra cena fica isolada. A comparação SIFT/ORB e a sensibilidade do ratio test expõem uma conexão fraca na extremidade da varredura.

O ajuste global reduz a inconsistência entre as vistas, e o cilindro torna a amplitude representável sem um plano único excessivamente distorcido. Costura e multibanda reduzem a mistura entre fontes na região estudada. O resultado ainda contém paralaxe e reflexos, visíveis principalmente em estruturas próximas e iluminadas.

Uma nova coleta diurna, com menor translação da câmera e mais sobreposição nas extremidades, permitiria avaliar se os mesmos parâmetros se mantêm. Não se trata de um panorama de 360°. A iluminação do conjunto atual deve ser apresentada como limitação em relação à recomendação de boa iluminação do T1.

## Referências

[R1] Lowe, D. G. (2004). *Distinctive Image Features from Scale-Invariant Keypoints*. IJCV 60(2), 91–110. [Artigo](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf).

[R2] Rublee, E. et al. (2011). *ORB: An Efficient Alternative to SIFT or SURF*. ICCV, 2564–2571. [Artigo](https://dev.ipol.im/~reyotero/bib/bib_all/2011_Rublee_Rabaud_ORB_alternative_to_sift_surf_willGarag_ICCV.pdf).

[R3] Fischler, M. A.; Bolles, R. C. (1981). *Random Sample Consensus*. CACM 24(6), 381–395. [Artigo](https://www.cs.ait.ac.th/~mdailey/cvreadings/Fischler-RANSAC.pdf).

[R4] Hartley, R.; Zisserman, A. (2004). *Multiple View Geometry in Computer Vision*, 2ª ed. Cambridge University Press. [Página dos autores](https://www.robots.ox.ac.uk/~vgg/hzbook/). Referência de aprofundamento; nesta preparação foram consultados os dados bibliográficos e o tratamento acessível em R6.

[R5] Brown, M.; Lowe, D. G. (2007). *Automatic Panoramic Image Stitching Using Invariant Features*. IJCV 74(1), 59–73. [Artigo](https://mattabrown.github.io/pdf/ijcv2007.pdf).

[R6] Szeliski, R. (2006). *Image Alignment and Stitching: A Tutorial*. MSR-TR-2004-92, revisão de 10/12/2006. [Relatório](https://pages.cs.wisc.edu/~dyer/ai-qual/szeliski-tr06.pdf).

[R7] Efros, A. A.; Freeman, W. T. (2001). *Image Quilting for Texture Synthesis and Transfer*. SIGGRAPH. [Artigo](https://pages.cs.wisc.edu/~dyer/cs534/papers/efros-freeman-siggraph01-quilting.pdf).

[R8] Burt, P. J.; Adelson, E. H. (1983). *A Multiresolution Spline With Application to Image Mosaics*. TOG 2(4), 217–236. [Artigo](https://ai.stanford.edu/~kosecka/burt-adelson-spline83.pdf).

[R9] Uyttendaele, M.; Eden, A.; Szeliski, R. (2001). *Eliminating Ghosting and Exposure Artifacts in Image Mosaics*. CVPR. [Artigo](https://www.cs.jhu.edu/~misha/ReadingSeminar/Papers/Uyttendaele01.pdf).

[R10] Agarwala, A. et al. (2004). *Interactive Digital Photomontage*. TOG 23(3), 294–302. [Artigo](https://grail.cs.washington.edu/projects/photomontage/photomontage.pdf). Fundamenta a alternativa por corte em grafo; não reproduzimos o sistema interativo completo.